1.Install

In [18]:
!pip install faiss-cpu pypdf numpy ollama

2. Imports

In [19]:
import os
import faiss
import pickle
import numpy as np
from pypdf import PdfReader
import ollama

3. Load Legal Documents (PDFs)

In [20]:
DATA_PATH = "rag_pdf"

def load_pdfs(folder):
    docs = []
    
    for file in os.listdir(folder):
        if file.endswith(".pdf"):
            reader = PdfReader(os.path.join(folder, file))
            
            for page in reader.pages:
                text = page.extract_text()
                if text:
                    docs.append(text)
    
    return docs

docs = load_pdfs(DATA_PATH)
print("Pages loaded:", len(docs))

Pages loaded: 6


4. Chunking

In [21]:
def chunk_text(text, size=800, overlap=150):
    chunks = []
    start = 0
    
    while start < len(text):
        chunk = text[start:start+size]
        chunks.append(chunk)
        start += size - overlap
    
    return chunks

all_chunks = []
for doc in docs:
    all_chunks.extend(chunk_text(doc))

print("Total chunks:", len(all_chunks))

Total chunks: 6


5. Pull Ollama models

In [22]:
!ollama pull nomic-embed-text
!ollama pull qwen

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest 
pulling 970aa74c0a90: 100% ▕██████████████████▏ 274 MB                         
pulling c71d239df917: 100% ▕██████████████████▏  11 KB                         
pulling ce4a164fc046: 100% ▕██████████████████▏   17 B                         
pulling 31df23ea7daa: 100% ▕██████████████████▏  420 B                         
verifying sha256 digest 
writing manifest 
success 
pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pu

6. Create embeddings

In [23]:
def get_embedding(text):
    res = ollama.embeddings(
        model="nomic-embed-text",
        prompt=text
    )
    return np.array(res["embedding"], dtype="float32")

embeddings = np.array([get_embedding(c) for c in all_chunks])

print("Embeddings shape:", embeddings.shape)

Embeddings shape: (6, 768)


7. Create FAISS index

In [24]:
dim = embeddings.shape[1]

faiss.normalize_L2(embeddings)

index = faiss.IndexFlatIP(dim)
index.add(embeddings)

print("FAISS index created")

FAISS index created


8. Save Index (for reuse)

In [25]:
os.makedirs("vectorstore", exist_ok=True)

faiss.write_index(index, "vectorstore/faiss_index.bin")

with open("vectorstore/metadata.pkl", "wb") as f:
    pickle.dump(all_chunks, f)

print("Saved successfully")

Saved successfully


9. Load index

In [26]:
index = faiss.read_index("vectorstore/faiss_index.bin")

with open("vectorstore/metadata.pkl", "rb") as f:
    all_chunks = pickle.load(f)

print("Loaded successfully")

Loaded successfully


10. Retrieval Function

In [27]:
def retrieve(query, k=4):
    q_vec = get_embedding(query).reshape(1, -1)
    faiss.normalize_L2(q_vec)

    scores, idx = index.search(q_vec, k)

    return [all_chunks[i] for i in idx[0]], scores[0]

11. RAG (FINAL LOGIC)

In [45]:
def ask_pdf(question):
    contexts, scores = retrieve(question)

    print("Similarity scores:", scores)  # debug

    # ✅ STRICT rejection
    if max(scores) < 0.30:
        return "❌ No relevant information found in the provided PDF documents."

    context_text = "\n\n".join(contexts)

    prompt = f"""
You are a legal assistant.

STRICT RULES:
- Answer ONLY from the provided context
- If the answer is not clearly in the context, respond EXACTLY with:
  "Not found in the provided documents."
- Do NOT guess
- Do NOT use prior knowledge
- Do NOT assume anything

Context:
{context_text}

Question:
{question}
"""

    res = ollama.chat(
        model="qwen",
        messages=[{"role": "user", "content": prompt}]
    )

    answer = res["message"]["content"]

    # ✅ Extra safety layer (post-check)
    if "not found" in answer.lower():
        return "❌ Not found in the provided PDF documents."

    return answer

12. Test

In [48]:
print(ask_pdf("Summarize the business agreement"))

Similarity scores: [0.65558165 0.6106874  0.5299775  0.514452  ]
The business agreement between ABC Traders and XYZ Enterprises outlines their collaboration in importing electronics products. Profit sharing ratio is set at 60% for ABC Traders and 40% for XYZ Enterprises. Confidentiality of suppliers and pricing are maintained by both parties. Agreement duration is set at 3 years starting from 2026. Either party may terminate with 60 days notice. Disputes will be resolved under Pakistan commercial law. Payment agreement outlines monthly rent of PKR 45,000 payable before 5th of each month along with security deposit of PKR 90,000 required. Tenant must not sublease the property without permission. Termination requires 30 days written notice.


Cell 12 — Advanced analysis

In [49]:
print(ask_pdf("What are the risks in this contract?"))

Similarity scores: [0.57925785 0.55295163 0.5234254  0.50306004]
The risks in this contract include:

1. Default: One party may fail to meet its obligations under the contract.

2. Breach of Contract: The other party may breach the terms of the contract, leading to disputes or legal action.

3. Force Majeure Events: unforeseen circumstances beyond the control of either party may cause delays or even cancellations of obligations under the contract.

4. Intellectual Property Rights: disputes over ownership or use rights of intellectual property may arise under the contract.

5. Taxation: disputes arising from differences in interpretation, application, and compliance with relevant tax laws and regulations between parties under the contract.


In [37]:
print(ask_pdf("Analyze this agreement and identify risks"))

Similarity scores: [0.6285604  0.5768337  0.5270788  0.52064896]
The rental agreement between Mr. Ali (Landlord) and Mr. Ahmed (Tenant) is as follows:

Monthly rent is 45,000 PKR payable before 5th of each month.

Security deposit of 90,000 PKR is required.

 Tenant must not sublease the property without permission.

Termination requires 30 days written notice.

In case of damage, tenant will bear repair costs.

Governing law: Pakistan LegalJurisdiction.

This rental agreement has certain risks associated with it. These risks include:

1. Risk of loss or damage to the property
2. Risk of breach of contract by the Landlord or Tenant
3. Risk of delay in payment of rent or security deposit
4. Risk of subleasing the property without permission, which could lead to legal action against the tenant

In conclusion, the rental agreement between Mr. Ali (Landlord) and Mr. Ahmed (Tenant) has certain risks associated with it.
